# Room Graph
- 1 node = 1 room
- 1 edge = shared door/window (access graph)

In [ ]:
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

print(Helper.Version())
renderer = "vscode"

## 1. Load geometry and analyse the cluster

In [ ]:
geo_objects = Topology.ByOBJPath("c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Geometry _etm.obj")
print(f"{len(geo_objects)} object(s) loaded")

cluster = Cluster.ByTopologies(geo_objects)
print(Topology.Analyze(cluster))

## 2. Build CellComplex
Use `ByCells` if the cluster contains cells (solids), otherwise `ByFaces`.

In [ ]:
cells = Topology.Cells(cluster)
faces = Topology.Faces(cluster)
print(f"Cells in cluster: {len(cells)}")
print(f"Faces in cluster: {len(faces)}")

if len(cells) > 0:
    print("Building CellComplex from cells...")
    cc_building = CellComplex.ByCells(cells)
else:
    print("No cells found — building CellComplex from faces...")
    cc_building = CellComplex.ByFaces(faces, tolerance=0.1)

print(f"CellComplex: {cc_building}")
if cc_building:
    print(f"Rooms detected: {len(Topology.Cells(cc_building))}")

## 3. Show the rooms

In [ ]:
Topology.Show(cc_building,
              faceOpacity=0.3,
              edgeColor="white",
              edgeWidth=1,
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)

## 4. Load apertures (doors & windows)

In [ ]:
apt_objects = Topology.ByOBJPath("c:/Users/etmaglari/IAAC/etmaglari_gML/Rhino_Apertures _etm.obj")
apt_cluster = Cluster.ByTopologies(apt_objects)
apertures = Topology.Faces(apt_cluster)
print(f"{len(apertures)} aperture face(s) loaded")

## 5. Add apertures to the CellComplex

In [ ]:
cc_building = Topology.AddApertures(cc_building, apertures)
print("Apertures added.")

## 6. Build the access graph

In [ ]:
g = Graph.ByTopology(cc_building, direct=False, viaSharedApertures=True)
print(f"Graph: {len(Graph.Vertices(g))} rooms, {len(Graph.Edges(g))} connections")

for v in Graph.Vertices(g):
    v = Topology.SetDictionary(v, Dictionary.ByKeysValues(["size","color"], [15, "red"]))
for e in Graph.Edges(g):
    e = Topology.SetDictionary(e, Dictionary.ByKeysValues(["width","color"], [3, "#2CFF96"]))

Topology.Show(cc_building, g,
              faceOpacity=0.05,
              vertexSizeKey="size", vertexColorKey="color",
              edgeWidthKey="width", edgeColorKey="color",
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)